In [ ]:
#问题1：编写代码，使用sklearn工具包中的make_classification函数，创建一个包含10个类别的不平衡数据集，样本数量、特征数量、类别分布、
#噪声状态由各个同学定义。统计各类别分布情况，绘制训练集合标签分布。

In [6]:
# 公共初始化代码｜依赖：无；输出：解释器、版本、运行目录
from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import sys
import uuid
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold

PROJECT_DIR = Path(r"D:\Python for Data Analysis\notebooks\imbalanced_classification")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), "请切换到 pda312 内核"
OUTPUT_BASE = Path(os.environ.get("ML_Q1_OUTPUT", str(PROJECT_DIR / "runs")))
RUN_DIR = OUTPUT_BASE / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
for folder in ("data", "results", "figures"):
    (RUN_DIR / folder).mkdir(parents=True, exist_ok=False)
VERSIONS = {p: metadata.version(p) for p in ("scikit-learn", "numpy", "pandas", "ipykernel")}
print("代码版本：v1.1；运行身份：", os.environ.get("ML_Q1_ACTOR", "学生运行，待学生核验"))
print("解释器：", sys.executable)
print("Python：", sys.version.split()[0], "；依赖：", VERSIONS)
print("本次输出目录：", RUN_DIR)

代码版本：v1.1；运行身份： 学生运行，待学生核验
解释器： D:\anaconda\envs\pda312\python.exe
Python： 3.12.13 ；依赖： {'scikit-learn': '1.9.0', 'numpy': '2.5.0', 'pandas': '3.0.3', 'ipykernel': '7.2.0'}
本次输出目录： D:\Python for Data Analysis\notebooks\imbalanced_classification\runs\20261020_221259_7f297c


In [2]:
LABELS = np.arange(10)
DATA_SEED, SPLIT_SEED, CV_SEED, MODEL_SEED = 1637, 20261020, 20261021, 20261022
TEST_SIZE, N_SPLITS = 0.25, 5
DATA_CONFIG = dict(
    n_samples=4800, n_features=24, n_informative=14,
    n_redundant=6, n_repeated=0, n_classes=10, n_clusters_per_class=2,
    weights=[0.32, 0.19, 0.13, 0.095, 0.075, 0.06, 0.045, 0.035, 0.03, 0.02],
    flip_y=0.015, class_sep=1.2, hypercube=True,
    shift=0.0, scale=1.0, shuffle=True, random_state=DATA_SEED,
)
assert len(DATA_CONFIG["weights"]) == 10 and np.isclose(sum(DATA_CONFIG["weights"]), 1)
assert min(DATA_CONFIG["weights"]) > 0
assert sum(DATA_CONFIG[k] for k in ("n_informative", "n_redundant", "n_repeated")) <= DATA_CONFIG["n_features"]
assert DATA_CONFIG["n_classes"] * DATA_CONFIG["n_clusters_per_class"] <= 2 ** DATA_CONFIG["n_informative"]
print(json.dumps(DATA_CONFIG, ensure_ascii=False, indent=2))

{
  "n_samples": 4800,
  "n_features": 24,
  "n_informative": 14,
  "n_redundant": 6,
  "n_repeated": 0,
  "n_classes": 10,
  "n_clusters_per_class": 2,
  "weights": [
    0.32,
    0.19,
    0.13,
    0.095,
    0.075,
    0.06,
    0.045,
    0.035,
    0.03,
    0.02
  ],
  "flip_y": 0.015,
  "class_sep": 1.2,
  "hypercube": true,
  "shift": 0.0,
  "scale": 1.0,
  "shuffle": true,
  "random_state": 1637
}


In [3]:
X, y = make_classification(**DATA_CONFIG)
assert np.array_equal(np.unique(y), LABELS), "生成数据未包含全部10类"
assert np.isfinite(X).all(), "特征存在非有限值"
# 先划分行号，便于保存并确保后续各题使用同一批样本。
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED
)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
train_counts = np.bincount(y_train, minlength=10)
test_counts = np.bincount(y_test, minlength=10)
assert (train_counts >= N_SPLITS).all() and (test_counts > 0).all()
assert len(np.intersect1d(train_idx, test_idx)) == 0
assert len(train_idx) + len(test_idx) == len(y)
print("完整数据形状：", X.shape, "；训练/测试样本数：", len(y_train), len(y_test))
print("训练集逐类计数：", train_counts)
print("测试集逐类计数：", test_counts)
print("10类齐全、样本划分无重叠、满足五折分层验证计数条件。")

完整数据形状： (4800, 24) ；训练/测试样本数： 3600 1200
训练集逐类计数： [1137  675  466  342  274  220  167  130  111   78]
测试集逐类计数： [379 225 155 114  91  74  56  43  37  26]
10类齐全、样本划分无重叠、满足五折分层验证计数条件。


In [4]:
distribution_parts = []
for split_name, split_y in (("all", y), ("train", y_train), ("test", y_test)):
    counts = np.bincount(split_y, minlength=10)
    distribution_parts.append(pd.DataFrame({
        "split": split_name, "class_label": LABELS,
        "count": counts, "proportion": counts / len(split_y),
        "target_weight": DATA_CONFIG["weights"],
    }))
distribution = pd.concat(distribution_parts, ignore_index=True)
train_distribution = distribution.query("split == 'train'").copy()
train_distribution["percent"] = train_distribution["proportion"] * 100
majority_class = int(np.flatnonzero(train_counts == train_counts.max())[0])
imbalance_ratio = float(train_counts.max() / train_counts.min())
display(train_distribution[["class_label", "count", "percent", "target_weight"]].round(4))
print(f"训练集不平衡比（最大计数/最小计数）：{imbalance_ratio:.4f}")
print(f"训练集多数类：{majority_class}；并列时选标签最小者")
print(f"训练集多数类占比：{train_counts.max() / len(y_train):.4%}")
print("以上是实际生成结果；目标权重不能替代实际计数。")

,class_label,count,percent,target_weight
10,0,1137,31.5833,0.320
11,1,675,18.7500,0.190
12,2,466,12.9444,0.130
13,3,342,9.5000,0.095
14,4,274,7.6111,0.075
15,5,220,6.1111,0.060
16,6,167,4.6389,0.045
17,7,130,3.6111,0.035
18,8,111,3.0833,0.030
19,9,78,2.1667,0.020


训练集不平衡比（最大计数/最小计数）：14.5769
训练集多数类：0；并列时选标签最小者
训练集多数类占比：31.5833%
以上是实际生成结果；目标权重不能替代实际计数。


In [7]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)
cv_rows, fold_indices = [], {}
for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    fold_indices[f"fold_{fold}_fit"] = fit_idx
    fold_indices[f"fold_{fold}_validation"] = val_idx
    for role, indices in (("fit", fit_idx), ("validation", val_idx)):
        counts = np.bincount(y_train[indices], minlength=10)
        assert (counts > 0).all(), "某个交叉验证折缺少类别"
        cv_rows.extend({"fold": fold, "role": role, "class_label": int(c), "count": int(n)}
                       for c, n in zip(LABELS, counts))
cv_counts = pd.DataFrame(cv_rows)
display(cv_counts.groupby(["fold", "role"])["count"].min().unstack().rename_axis(columns=None))
print("表内为各折最少类别计数；五折的拟合部分和验证部分均包含10类。")

distribution.to_csv(RUN_DIR / "results/class_distribution.csv", index=False, encoding="utf-8-sig")
cv_counts.to_csv(RUN_DIR / "results/cv_class_counts.csv", index=False, encoding="utf-8-sig")
np.savez_compressed(RUN_DIR / "data/dataset_and_split.npz", X=X, y=y,
                    train_idx=train_idx, test_idx=test_idx, **fold_indices)
features = pd.DataFrame(X, columns=[f"feature_{i:02d}" for i in range(X.shape[1])])
features.insert(0, "sample_id", np.arange(len(y)))
features["label"] = y
features["split"] = np.where(np.isin(np.arange(len(y)), train_idx), "train", "test")
features.to_csv(RUN_DIR / "data/dataset.csv", index=False, encoding="utf-8-sig")
data_hash = hashlib.sha256(X.tobytes() + y.tobytes() + train_idx.tobytes() + test_idx.tobytes()).hexdigest()
manifest = dict(
    code_version="1.1", configuration_status="建议配置；学生采用与理解待确认",
    run_actor=os.environ.get("ML_Q1_ACTOR", "学生运行，待学生核验"),
    created_at=datetime.now().astimezone().isoformat(), python=sys.version,
    executable=sys.executable, versions=VERSIONS, make_classification=DATA_CONFIG,
    test_size=TEST_SIZE, split_seed=SPLIT_SEED, cv_seed=CV_SEED, model_seed=MODEL_SEED,
    cv_folds=N_SPLITS, labels=LABELS.tolist(), zero_division=0,
    future_metrics=["accuracy", "macro_f1"], train_majority_class=majority_class,
    train_imbalance_ratio=imbalance_ratio, data_split_sha256=data_hash,
    cv_index_basis="positions within X_train, not original X", output_dir=str(RUN_DIR),
)
(RUN_DIR / "results/manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("数据与划分SHA256：", data_hash)
print("结果已保存到：", RUN_DIR)

,fit,validation
fold,,
1,63,15
2,63,15
3,62,16
4,62,16
5,62,16


表内为各折最少类别计数；五折的拟合部分和验证部分均包含10类。
数据与划分SHA256： 267db14403d8885a89d6396fa644614f8ec2a06e60a3c45ac35e508b6f6723d4
结果已保存到： D:\Python for Data Analysis\notebooks\imbalanced_classification\runs\20261020_221259_7f297c
